# Analisis PCA - Dataset RRHH

Objetivo: determinar si PCA es una tecnica adecuada para reducir la dimensionalidad del dataset antes de aplicar clustering. Evaluaremos:
1. Preparacion de datos (escalado + encoding)
2. Matriz de correlacion entre variables numericas
3. Varianza explicada por cada componente principal
4. Decision: merece la pena aplicar PCA?

## 1. Carga de datos y exploracion inicial

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

df = pd.read_csv('dataSet_RRHH.csv')
print(f"Shape: {df.shape}")
print(f"\nTipos de datos:\n{df.dtypes}")
print(f"\nPrimeras filas:")
df.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Estadisticas descriptivas de las variables numericas
df.describe().round(2)

## 2. Preprocesamiento

Para PCA necesitamos que todas las variables sean numericas y esten en la misma escala:
- **Variables numericas**: escalamos con `StandardScaler` (media=0, std=1)
- **Variables categoricas**: aplicamos one-hot encoding y luego escalamos tambien

In [ ]:
# Separar variables numericas y categoricas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Variables numericas ({len(num_cols)}): {num_cols}")
print(f"Variables categoricas ({len(cat_cols)}): {cat_cols}")
for c in cat_cols:
    print(f"  {c}: {df[c].unique().tolist()}")

In [ ]:
# One-hot encoding de categoricas (drop_first para evitar multicolinealidad)
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
print(f"Shape tras encoding: {df_encoded.shape}")
print(f"Columnas: {df_encoded.columns.tolist()}")

In [ ]:
# Escalado con StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)
X_scaled = pd.DataFrame(X_scaled, columns=df_encoded.columns)

print("Verificacion tras escalado (media ~ 0, std ~ 1):")
print(X_scaled.describe().loc[['mean', 'std']].round(4))

## 3. Matriz de correlacion

Antes de PCA, veamos si hay correlaciones fuertes entre variables. Si las variables son muy independientes entre si, PCA tendra poco que comprimir.

In [ ]:
# Matriz de correlacion
corr_matrix = X_scaled.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Matriz de Correlacion (variables escaladas)', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# Correlaciones mas fuertes (en valor absoluto)
corr_pairs = corr_matrix.where(mask == False).stack().reset_index()
corr_pairs.columns = ['Var1', 'Var2', 'Correlacion']
corr_pairs['abs_corr'] = corr_pairs['Correlacion'].abs()
print("\nTop 10 correlaciones mas fuertes:")
print(corr_pairs.nlargest(10, 'abs_corr')[['Var1', 'Var2', 'Correlacion']].to_string(index=False))

## 4. PCA - Analisis de componentes principales

Aplicamos PCA con **todos** los componentes para ver cuanta varianza explica cada uno. Esto nos permite decidir si merece la pena reducir dimensiones.

In [ ]:
# PCA con todos los componentes
pca_full = PCA()
pca_full.fit(X_scaled)

# Varianza explicada por cada componente
var_explicada = pca_full.explained_variance_ratio_
var_acumulada = np.cumsum(var_explicada)

n_components = len(var_explicada)

# Tabla resumen
tabla_pca = pd.DataFrame({
    'Componente': [f'PC{i+1}' for i in range(n_components)],
    'Autovalor': pca_full.explained_variance_,
    'Varianza Explicada (%)': var_explicada * 100,
    'Varianza Acumulada (%)': var_acumulada * 100
}).round(4)

print(f"Numero total de componentes: {n_components}")
print(f"\n{'='*60}")
print(tabla_pca.to_string(index=False))
print(f"{'='*60}")

### 4.1 Scree Plot y Varianza Acumulada

El **scree plot** muestra la varianza explicada por cada componente. Buscamos un "codo" donde la ganancia marginal cae drasticamente. La linea de varianza acumulada nos dice cuantos componentes necesitamos para capturar un % deseado (tipicamente 80-90%).

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 7))

# Barras: varianza individual
x = np.arange(1, n_components + 1)
bars = ax1.bar(x, var_explicada * 100, color='steelblue', alpha=0.8, label='Varianza individual')

# Etiquetas sobre las barras
for bar, val in zip(bars, var_explicada * 100):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax1.set_xlabel('Componente Principal', fontsize=13)
ax1.set_ylabel('Varianza Explicada (%)', fontsize=13, color='steelblue')
ax1.set_xticks(x)
ax1.set_xticklabels([f'PC{i}' for i in x])
ax1.tick_params(axis='y', labelcolor='steelblue')

# Linea: varianza acumulada
ax2 = ax1.twinx()
ax2.plot(x, var_acumulada * 100, 'o-', color='firebrick', linewidth=2.5,
         markersize=8, label='Varianza acumulada')

# Etiquetas en la linea acumulada
for xi, val in zip(x, var_acumulada * 100):
    ax2.annotate(f'{val:.1f}%', (xi, val), textcoords="offset points",
                 xytext=(0, 12), ha='center', fontsize=10, color='firebrick', fontweight='bold')

ax2.set_ylabel('Varianza Acumulada (%)', fontsize=13, color='firebrick')
ax2.tick_params(axis='y', labelcolor='firebrick')
ax2.set_ylim(0, 105)

# Lineas de referencia
for threshold in [80, 90]:
    ax2.axhline(y=threshold, color='gray', linestyle='--', alpha=0.5)
    ax2.text(n_components + 0.3, threshold, f'{threshold}%', va='center', fontsize=10, color='gray')

# Criterio de Kaiser: autovalor > 1
n_kaiser = sum(pca_full.explained_variance_ > 1)
ax1.axvline(x=n_kaiser + 0.5, color='green', linestyle=':', linewidth=2, alpha=0.7)
ax1.text(n_kaiser + 0.6, max(var_explicada * 100) * 0.9,
         f'Kaiser: {n_kaiser} comp.', color='green', fontsize=11)

ax1.set_title('Scree Plot - Varianza Explicada por Componente Principal', fontsize=15, pad=15)

# Leyenda combinada
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nCriterio de Kaiser (autovalor > 1): retener {n_kaiser} componentes")
print(f"  -> Varianza acumulada con {n_kaiser} componentes: {var_acumulada[n_kaiser-1]*100:.1f}%")

# Componentes para 80% y 90%
for threshold in [0.80, 0.90]:
    n_thresh = np.argmax(var_acumulada >= threshold) + 1
    print(f"  -> Componentes para >={threshold*100:.0f}% varianza: {n_thresh}")

### 4.2 Peso de cada variable original en los componentes principales

Para interpretar que representa cada componente, vemos los loadings (pesos) de las variables originales.

In [ ]:
# Loadings (pesos de cada variable en cada componente)
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=[f'PC{i+1}' for i in range(n_components)],
    index=df_encoded.columns
)

# Mostrar los loadings de los primeros componentes relevantes
n_show = min(n_components, 6)
print(f"Loadings de los primeros {n_show} componentes:\n")
print(loadings.iloc[:, :n_show].round(3).to_string())

In [ ]:
# Heatmap de loadings para los primeros componentes
n_show = min(n_components, 6)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(loadings.iloc[:, :n_show], annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title(f'Loadings - Peso de cada variable en los primeros {n_show} componentes', fontsize=13, pad=15)
ax.set_xlabel('Componente Principal')
ax.set_ylabel('Variable Original')
plt.tight_layout()
plt.show()

## 5. Conclusion: Merece la pena aplicar PCA?

Criterios para decidir:

| Criterio | A favor de PCA | En contra de PCA |
|---|---|---|
| **Varianza acumulada** | Pocos componentes capturan >80% | Necesitas casi todos para >80% |
| **Scree plot** | Codo claro y pronunciado | Caida gradual y uniforme |
| **Kaiser** | Pocos componentes con autovalor >1 | Muchos componentes con autovalor >1 |
| **Correlaciones** | Variables correlacionadas (redundancia) | Variables independientes |
| **Dimensionalidad** | Muchas variables originales | Pocas variables originales |

**Nota**: Con solo 12 variables totales (6 num + 6 dummies tras encoding), la reduccion dimensional solo merece la pena si hay correlaciones fuertes que PCA pueda aprovechar. Ejecuta las celdas anteriores y usa la tabla y graficas para llegar a tu conclusion.

In [ ]:
# Resumen automatico de la decision
print("=" * 60)
print("RESUMEN AUTOMATICO")
print("=" * 60)

# Cuantos componentes para 80%?
n_80 = np.argmax(var_acumulada >= 0.80) + 1
ratio_reduccion = 1 - (n_80 / n_components)

print(f"\n- Variables originales (tras encoding): {n_components}")
print(f"- Componentes para >=80% varianza: {n_80} (reduccion del {ratio_reduccion*100:.0f}%)")
print(f"- Componentes con autovalor >1 (Kaiser): {n_kaiser}")
print(f"- Correlacion maxima entre variables: {corr_pairs.nlargest(1, 'abs_corr')['Correlacion'].values[0]:.3f}")

print(f"\n{'='*60}")
if n_80 <= n_components * 0.6 and ratio_reduccion >= 0.3:
    print("VEREDICTO: PCA PUEDE SER UTIL")
    print(f"  Se puede reducir de {n_components} a {n_80} componentes")
    print(f"  manteniendo >=80% de la varianza ({ratio_reduccion*100:.0f}% reduccion).")
else:
    print("VEREDICTO: PCA PROBABLEMENTE NO MERECE LA PENA")
    print(f"  Para retener >=80% varianza necesitas {n_80} de {n_components} componentes.")
    print(f"  La reduccion ({ratio_reduccion*100:.0f}%) es minima y se pierde interpretabilidad.")
    print("  Considera usar las variables originales directamente para clustering.")
print("=" * 60)